# Data Generator – Car Workshop & Accessories Shop Network

**Business scenario:** Network of 100 car workshops and accessories shops across Poland.  
**Period:** 2020-01 to 2026-12 (7 years)  
**Scale:** ~10-50 GB (configurable via SCALE_FACTOR)  

## Tables:
### Dimension
1. `dim_locations` - locations (100)
2. `dim_employees` - employees (~2000)
3. `dim_customers` - customers (~500K)
4. `dim_vehicles` - vehicles (~600K)
5. `dim_products` - products/parts (~15K)
6. `dim_services` - service catalogue (~200)
7. `dim_suppliers` - suppliers (~300)

### Fact
8. `fact_work_orders` - workshop work orders (~5M)
9. `fact_work_order_items` - work order items (~15M)
10. `fact_sales_transactions` - retail sales transactions (~30M)
11. `fact_sales_items` - sales items (~90M)
12. `fact_invoices` - invoices (~35M)
13. `fact_payments` - payments (~35M)
14. `fact_inventory_movements` - inventory movements (~50M)

### Supporting
15. `fact_appointments` - bookings (~5M)
16. `fact_purchase_orders` - supplier orders (~500K)
17. `fact_purchase_order_items` - purchase order items (~2M)
18. `fact_customer_feedback` - reviews (~2M)
19. `fact_loyalty_program` - loyalty programme (~500K)
20. `fact_employee_schedules` - work schedules (~3M)

In [0]:
# Instalacja zależności
!pip install pandas pyarrow faker tqdm

In [0]:
import os
import uuid
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import datetime, timedelta, date
from faker import Faker
from tqdm import tqdm
import random
import uuid
import gc
import json

fake = Faker('pl_PL')
Faker.seed(42)
np.random.seed(42)
random.seed(42)

print('Biblioteki załadowane OK')

In [0]:
# ============================================================
# KONFIGURACJA
# ============================================================

# SCALE_FACTOR: 1.0 = pełne dane (~30GB), 0.1 = ~3GB, 0.01 = ~300MB do testów
SCALE_FACTOR = 1
#
# Katalog wyjściowy
FACT_OUTPUT_DIR = '/Volumes/car_workshop/fact/fact_files'
DIM_OUTPUT_DIR = '/Volumes/car_workshop/dim/dim_files'
# Format: 'parquet' lub 'csv'
OUTPUT_FORMAT = 'parquet'

# FACT_OUTPUT_DIR = './output/fact_daily_files'
# DIM_OUTPUT_DIR = './output/dim_daily_files'

# Okres danych
DATE_END = date.today()
DATE_START = DATE_END - timedelta(days=1)

# Rozmiar chunka przy zapisie (wiersze)
CHUNK_SIZE = 128_000

# Liczba lokalizacji
NUM_LOCATIONS = 100

os.makedirs(FACT_OUTPUT_DIR, exist_ok=True)
os.makedirs(DIM_OUTPUT_DIR, exist_ok=True)
print(f'SCALE_FACTOR = {SCALE_FACTOR}')
print(f'OUTPUT_FACT_DIR = {FACT_OUTPUT_DIR}')
print(f'OUTPUT_DIM_DIR = {DIM_OUTPUT_DIR}')
print(f'Szacowana wielkość danych: ~{SCALE_FACTOR * 30:.1f} GB')

In [0]:
# ============================================================
# HELPERY
# ============================================================

def save_table(df, table_name, partition_cols=None):
    """Zapisuje DataFrame jako parquet lub csv."""
    table_dir = os.path.join(DIM_OUTPUT_DIR, table_name)
    os.makedirs(table_dir, exist_ok=True)
    
    if OUTPUT_FORMAT == 'parquet':
        if partition_cols:
            pq.write_to_dataset(
                pa.Table.from_pandas(df),
                root_path=table_dir,
                partition_cols=partition_cols
            )
        else:
            pq.write_table(
                pa.Table.from_pandas(df),
                os.path.join(table_dir, f'{table_name}.parquet')
            )
    else:
        df.to_csv(os.path.join(table_dir, f'{table_name}.csv'), index=False)
    
    size_mb = df.memory_usage(deep=True).sum() / 1024 / 1024
    print(f'  ✓ {table_name}: {len(df):,} wierszy, ~{size_mb:.1f} MB w pamięci')




def save_table_chunked(generate_func, table_name, total_rows, dir, partition_cols=None):
    """Generuje i zapisuje dane w chunkach aby oszczędzić RAM."""
    table_dir = os.path.join(dir, table_name)
    os.makedirs(table_dir, exist_ok=True)
    
    rows_written = 0
    chunk_num = 0
    
    with tqdm(total=total_rows, desc=table_name) as pbar:
        while rows_written < total_rows:
            chunk_rows = min(CHUNK_SIZE, total_rows - rows_written)
            df_chunk = generate_func(chunk_rows, rows_written)
            
            if OUTPUT_FORMAT == 'parquet':
                if partition_cols:
                    pq.write_to_dataset(
                        pa.Table.from_pandas(df_chunk),
                        root_path=table_dir,
                        partition_cols=partition_cols
                    )
                else:
                    chunk_id = uuid.uuid4().hex[:9]  # 9-znakowy hex, np. 'a3f9c1b27'
                    pq.write_table(
                        pa.Table.from_pandas(df_chunk),
                        os.path.join(table_dir, f'{table_name}_part_{chunk_id}.parquet')
                    )
            else:
                mode = 'w' if chunk_num == 0 else 'a'
                header = chunk_num == 0
                df_chunk.to_csv(
                    os.path.join(table_dir, f'{table_name}.csv'),
                    index=False, mode=mode, header=header
                )
            
            rows_written += chunk_rows
            chunk_num += 1
            pbar.update(chunk_rows)
            del df_chunk
            gc.collect()
    
    print(f'  ✓ {table_name}: {rows_written:,} wierszy w {chunk_num} chunkach')


def random_dates(start, end, n):
    """Generuje n losowych dat z zakresu z uwzględnieniem sezonowości."""
    start_ts = pd.Timestamp(start)
    end_ts = pd.Timestamp(end)
    delta = (end_ts - start_ts).days
    random_days = np.random.randint(0, delta, size=n)
    dates = start_ts + pd.to_timedelta(random_days, unit='D')
    return dates


def seasonal_dates(start, end, n):
    """Generuje daty z sezonowością - więcej w okresach jesień/wiosna."""
    dates = random_dates(start, end, n)
    months = dates.month
    # Wagi sezonowe: więcej w marcu-kwietniu (wymiana opon) i październiku-listopadzie
    seasonal_weights = {1: 0.7, 2: 0.7, 3: 1.4, 4: 1.4, 5: 1.0, 6: 0.9,
                        7: 0.8, 8: 0.8, 9: 1.0, 10: 1.4, 11: 1.3, 12: 0.6}
    weights = np.array([seasonal_weights[m] for m in months])
    weights = weights / weights.sum()
    indices = np.random.choice(len(dates), size=n, replace=True, p=weights)
    return dates[indices]


def generate_uuid_batch(n):
    """Generuje batch UUID-ów."""
    return [str(uuid.uuid4()) for _ in range(n)]


print('Helpery załadowane OK')

## 1. Dane referencyjne (słowniki)

In [0]:
# ============================================================
# REFERENCE DATA DICTIONARIES
# ============================================================

REGIONS = [
    'Lower Silesian', 'Kuyavian-Pomeranian', 'Lublin', 'Lubusz',
    'Lodz', 'Lesser Poland', 'Masovian', 'Opole',
    'Subcarpathian', 'Podlaskie', 'Pomeranian', 'Silesian',
    'Holy Cross', 'Warmian-Masurian', 'Greater Poland', 'West Pomeranian'
]

CITIES = [
    ('Warsaw',               'Masovian',              52.2297, 21.0122),
    ('Krakow',               'Lesser Poland',         50.0647, 19.9450),
    ('Lodz',                 'Lodz',                  51.7592, 19.4560),
    ('Wroclaw',              'Lower Silesian',        51.1079, 17.0385),
    ('Poznan',               'Greater Poland',        52.4064, 16.9252),
    ('Gdansk',               'Pomeranian',            54.3520, 18.6466),
    ('Szczecin',             'West Pomeranian',       53.4285, 14.5528),
    ('Bydgoszcz',            'Kuyavian-Pomeranian',   53.1235, 18.0084),
    ('Lublin',               'Lublin',                51.2465, 22.5684),
    ('Bialystok',            'Podlaskie',             53.1325, 23.1688),
    ('Katowice',             'Silesian',              50.2649, 19.0238),
    ('Gdynia',               'Pomeranian',            54.5189, 18.5305),
    ('Czestochowa',          'Silesian',              50.8118, 19.1203),
    ('Radom',                'Masovian',              51.4027, 21.1471),
    ('Sosnowiec',            'Silesian',              50.2863, 19.1041),
    ('Torun',                'Kuyavian-Pomeranian',   53.0138, 18.5984),
    ('Kielce',               'Holy Cross',            50.8661, 20.6286),
    ('Rzeszow',              'Subcarpathian',         50.0412, 21.9991),
    ('Gliwice',              'Silesian',              50.2945, 18.6714),
    ('Zabrze',               'Silesian',              50.3249, 18.7857),
    ('Olsztyn',              'Warmian-Masurian',      53.7784, 20.4801),
    ('Bielsko-Biala',        'Silesian',              49.8224, 19.0586),
    ('Bytom',                'Silesian',              50.3483, 18.9157),
    ('Zielona Gora',         'Lubusz',                51.9356, 15.5062),
    ('Rybnik',               'Silesian',              50.1022, 18.5463),
    ('Ruda Slaska',          'Silesian',              50.2558, 18.8556),
    ('Opole',                'Opole',                 50.6751, 17.9213),
    ('Tychy',                'Silesian',              50.1357, 18.9936),
    ('Gorzow Wielkopolski',  'Lubusz',                52.7325, 15.2369),
    ('Elblag',               'Warmian-Masurian',      54.1522, 19.4088),
    ('Plock',                'Masovian',              52.5463, 19.7065),
    ('Dabrowa Gornicza',     'Silesian',              50.3217, 19.1880),
    ('Walbrzych',            'Lower Silesian',        50.7714, 16.2843),
    ('Wloclawek',            'Kuyavian-Pomeranian',   52.6483, 19.0677),
    ('Tarnow',               'Lesser Poland',         50.0121, 20.9858),
    ('Chorzow',              'Silesian',              50.2975, 18.9545),
    ('Koszalin',             'West Pomeranian',       54.1943, 16.1715),
    ('Kalisz',               'Greater Poland',        51.7611, 18.0909),
    ('Legnica',              'Lower Silesian',        51.2070, 16.1619),
    ('Grudziadz',            'Kuyavian-Pomeranian',   53.4837, 18.7536),
    ('Jaworzno',             'Silesian',              50.2040, 19.2747),
    ('Slupsk',               'Pomeranian',            54.4641, 17.0285),
    ('Jastrzebie-Zdroj',     'Silesian',              49.9477, 18.5963),
    ('Nowy Sacz',            'Lesser Poland',         49.6249, 20.6915),
    ('Jelenia Gora',         'Lower Silesian',        50.9044, 15.7197),
    ('Siedlce',              'Masovian',              52.1676, 22.2903),
    ('Myslowice',            'Silesian',              50.2083, 19.1666),
    ('Konin',                'Greater Poland',        52.2230, 18.2511),
    ('Pila',                 'Greater Poland',        53.1510, 16.7382),
    ('Piotrkow Trybunalski', 'Lodz',                  51.4053, 19.7031),
    ('Inowroclaw',           'Kuyavian-Pomeranian',   52.7936, 18.2614),
    ('Lubin',                'Lower Silesian',        51.4010, 16.2015),
    ('Ostrow Wielkopolski',  'Greater Poland',        51.6550, 17.8068),
    ('Suwalki',              'Podlaskie',             54.1118, 22.9308),
    ('Stargard',             'West Pomeranian',       53.3364, 15.0502),
    ('Gniezno',              'Greater Poland',        52.5348, 17.5827),
    ('Ostrowiec Swietokrzyski', 'Holy Cross',         50.9295, 21.3856),
    ('Siemianowice Slaskie', 'Silesian',              50.3264, 19.0296),
    ('Glogow',               'Lower Silesian',        51.6634, 16.0845),
    ('Pabianice',            'Lodz',                  51.6649, 19.3548),
    ('Leszno',               'Greater Poland',        51.8425, 16.5749),
    ('Zory',                 'Silesian',              50.0455, 18.7005),
    ('Pruszkow',             'Masovian',              52.1707, 20.8120),
    ('Stalowa Wola',         'Subcarpathian',         50.5828, 22.0531),
    ('Zamosc',               'Lublin',                50.7230, 23.2519),
    ('Lomza',                'Podlaskie',             53.1784, 22.0593),
    ('Mielec',               'Subcarpathian',         50.2874, 21.4260),
    ('Tczew',                'Pomeranian',            54.0927, 18.7955),
    ('Chelm',                'Lublin',                51.1431, 23.4716),
    ('Przemysl',             'Subcarpathian',         49.7838, 22.7678),
    ('Starachowice',         'Holy Cross',            51.0378, 21.0714),
    ('Wejherowo',            'Pomeranian',            54.6059, 18.2354),
    ('Pulawy',               'Lublin',                51.4166, 21.9686),
    ('Skierniewice',         'Lodz',                  51.9542, 20.1576),
    ('Skarzysko-Kamienna',   'Holy Cross',            51.1141, 20.8597),
    ('Tarnobrzeg',           'Subcarpathian',         50.5731, 21.6792),
    ('Radomsko',             'Lodz',                  51.0671, 19.4462),
    ('Kedzierzyn-Kozle',     'Opole',                 50.3494, 18.2074),
    ('Biala Podlaska',       'Lublin',                52.0326, 23.1166),
    ('Oswiecim',             'Lesser Poland',         50.0343, 19.2098),
    ('Sandomierz',           'Holy Cross',            50.6827, 21.7489),
    ('Busko-Zdroj',          'Holy Cross',            50.4710, 20.7192),
    ('Nowa Sol',             'Lubusz',                51.8063, 15.7146),
    ('Nysa',                 'Opole',                 50.4743, 17.3346),
    ('Otwock',               'Masovian',              52.1054, 21.2614),
    ('Szczytno',             'Warmian-Masurian',      53.5630, 20.9868),
    ('Kutno',                'Lodz',                  52.2318, 19.3569),
    ('Sanok',                'Subcarpathian',         49.5566, 22.2059),
    ('Swinoujscie',          'West Pomeranian',       53.9101, 14.2474),
    ('Swidnica',             'Lower Silesian',        50.8463, 16.4872),
    ('Chojnice',             'Pomeranian',            53.6953, 17.5551),
    ('Minsk Mazowiecki',     'Masovian',              52.1790, 21.5617),
    ('Zyrardow',             'Masovian',              52.0491, 20.4467),
    ('Wolomin',              'Masovian',              52.3461, 21.2405),
    ('Nowy Targ',            'Lesser Poland',         49.4782, 20.0323),
    ('Gizycko',              'Warmian-Masurian',      54.0380, 21.7647),
    ('Brodnica',             'Kuyavian-Pomeranian',   53.2600, 19.3954),
    ('Boleslawiec',          'Lower Silesian',        51.2622, 15.5694),
    ('Swiecie',              'Kuyavian-Pomeranian',   53.4100, 18.4316),
]

CAR_MAKES = {
    'Toyota': ['Corolla', 'Yaris', 'RAV4', 'Camry', 'C-HR', 'Aygo', 'Hilux', 'Land Cruiser'],
    'Volkswagen': ['Golf', 'Passat', 'Polo', 'Tiguan', 'T-Roc', 'Arteon', 'Touran', 'Caddy'],
    'Skoda': ['Octavia', 'Fabia', 'Superb', 'Kodiaq', 'Karoq', 'Kamiq', 'Scala', 'Citigo'],
    'Ford': ['Focus', 'Fiesta', 'Mondeo', 'Kuga', 'Puma', 'EcoSport', 'Transit', 'Ranger'],
    'Opel': ['Astra', 'Corsa', 'Insignia', 'Mokka', 'Crossland', 'Grandland', 'Combo', 'Vivaro'],
    'BMW': ['Series 3', 'Series 5', 'X1', 'X3', 'Series 1', 'X5', 'Series 7', 'X6'],
    'Audi': ['A3', 'A4', 'A6', 'Q3', 'Q5', 'A1', 'Q7', 'TT'],
    'Mercedes': ['Class A', 'Class C', 'Class E', 'GLC', 'GLA', 'GLE', 'Class S', 'Sprinter'],
    'Renault': ['Clio', 'Megane', 'Captur', 'Kadjar', 'Scenic', 'Kangoo', 'Master', 'Trafic'],
    'Hyundai': ['i30', 'Tucson', 'i20', 'Kona', 'Santa Fe', 'i10', 'ix20', 'Ioniq'],
    'Kia': ['Ceed', 'Sportage', 'Rio', 'Stonic', 'Sorento', 'Picanto', 'XCeed', 'Niro'],
    'Fiat': ['500', 'Tipo', 'Panda', 'Punto', '500X', 'Ducato', 'Doblo', '500L'],
    'Peugeot': ['208', '308', '3008', '2008', '508', '5008', 'Partner', 'Rifter'],
    'Citroen': ['C3', 'C4', 'C5 Aircross', 'Berlingo', 'C3 Aircross', 'C1', 'Jumper', 'Jumpy'],
    'Dacia': ['Duster', 'Sandero', 'Logan', 'Dokker', 'Lodgy', 'Spring'],
    'Nissan': ['Qashqai', 'Juke', 'Micra', 'X-Trail', 'Navara', 'Leaf', 'Note'],
    'Honda': ['Civic', 'CR-V', 'Jazz', 'HR-V', 'Accord', 'e'],
    'Mazda': ['3', '6', 'CX-5', 'CX-3', 'CX-30', 'MX-5', '2'],
    'Volvo': ['XC60', 'XC40', 'V60', 'S60', 'XC90', 'V40', 'S90'],
    'Suzuki': ['Vitara', 'Swift', 'SX4 S-Cross', 'Ignis', 'Jimny', 'Baleno'],
}

# Brand popularity weights for Poland
MAKE_WEIGHTS = {
    'Toyota': 0.12, 'Volkswagen': 0.11, 'Skoda': 0.10, 'Ford': 0.08,
    'Opel': 0.08, 'BMW': 0.05, 'Audi': 0.05, 'Mercedes': 0.04,
    'Renault': 0.06, 'Hyundai': 0.06, 'Kia': 0.06, 'Fiat': 0.04,
    'Peugeot': 0.04, 'Citroen': 0.03, 'Dacia': 0.03, 'Nissan': 0.02,
    'Honda': 0.02, 'Mazda': 0.02, 'Volvo': 0.02, 'Suzuki': 0.02,
}

FUEL_TYPES = ['petrol', 'diesel', 'LPG', 'hybrid', 'electric']
FUEL_WEIGHTS = [0.35, 0.30, 0.15, 0.15, 0.05]

COLORS = ['white', 'black', 'silver', 'grey', 'red', 'blue',
          'navy', 'green', 'brown', 'beige', 'gold', 'maroon']

PRODUCT_CATEGORIES = {
    'Oils and Fluids': [
        'Engine oil 5W-30', 'Engine oil 5W-40', 'Engine oil 10W-40',
        'Engine oil 0W-20', 'Brake fluid DOT4', 'Coolant G12',
        'Summer windscreen wash', 'Winter windscreen wash',
        'Gearbox oil', 'Power steering fluid', 'AdBlue fluid 10L',
    ],
    'Filters': [
        'Oil filter', 'Air filter', 'Cabin filter', 'Fuel filter',
        'Active carbon cabin filter', 'DPF filter', 'GPF filter',
    ],
    'Brake Pads and Discs': [
        'Brake pads front', 'Brake pads rear',
        'Brake discs front', 'Brake discs rear',
        'Brake shoes', 'Brake drums',
    ],
    'Tyres': [
        'Summer tyre 205/55 R16', 'Summer tyre 195/65 R15',
        'Summer tyre 225/45 R17', 'Winter tyre 205/55 R16',
        'Winter tyre 195/65 R15', 'Winter tyre 225/45 R17',
        'All-season tyre 205/55 R16', 'All-season tyre 195/65 R15',
    ],
    'Batteries': [
        'Battery 60Ah', 'Battery 70Ah', 'Battery 74Ah',
        'Battery 80Ah', 'Battery 100Ah', 'AGM Battery 70Ah',
    ],
    'Lighting': [
        'H7 bulb', 'H4 bulb', 'H1 bulb', 'LED H7 bulb',
        'LED H4 bulb', 'W5W bulb', 'P21W bulb',
        'D1S xenon bulb', 'D2S xenon bulb',
    ],
    'Wipers': [
        'Front left wiper blade', 'Front right wiper blade',
        'Rear wiper blade', 'Front wiper blade set',
    ],
    'Suspension System': [
        'Front shock absorber', 'Rear shock absorber', 'Suspension spring',
        'Lower control arm', 'Stabiliser link', 'Control arm bushing',
        'Tie rod end', 'Tie rod',
    ],
    'Timing System': [
        'Timing belt', 'Timing kit with water pump',
        'Multi-V belt', 'Timing belt tensioner',
        'Timing chain', 'Timing chain kit',
    ],
    'Exhaust System': [
        'Rear muffler', 'Middle muffler', 'Catalytic converter',
        'Exhaust pipe', 'DPF particulate filter',
        'Lambda sensor', 'Exhaust gasket',
    ],
    'Electrical System': [
        'Alternator', 'Starter motor', 'Ignition coil',
        'Spark plug', 'Glow plug', 'ABS sensor',
        'Temperature sensor', 'Oil pressure sensor',
    ],
    'Clutch': [
        'Clutch kit', 'Clutch disc', 'Clutch pressure plate',
        'Clutch release bearing', 'Dual mass flywheel',
    ],
    'Car Care Products': [
        'Car shampoo', 'Paint wax', 'Wheel cleaner',
        'De-icer', 'Air freshener', 'Polishing compound',
        'Upholstery cleaner', 'Seal silicone',
        'Plastic restorer', 'Anti-corrosion spray',
    ],
    'Accessories': [
        'Rubber floor mat set', 'Velour floor mat set',
        'Seat covers', 'Boot organiser', 'First aid kit',
        'Warning triangle', 'Car fire extinguisher', 'Jump leads',
        'Car compass', 'USB car charger',
        'Phone holder', 'Reversing camera', 'Parking sensors',
        'Roof rack', 'Roof box', 'Tow bar',
    ],
    'Tools': [
        'Wheel wrench', 'Hydraulic jack', 'Socket wrench set',
        'Torque wrench', 'Tyre repair kit',
    ],
}

SERVICE_CATALOGUE = [
    # (name, category, min_price_net, max_price_net, estimated_time_min)
    ('Oil and filter change', 'Periodic Service', 80, 150, 30),
    ('Periodic service', 'Periodic Service', 150, 350, 60),
    ('Air filter replacement', 'Periodic Service', 30, 60, 15),
    ('Cabin filter replacement', 'Periodic Service', 30, 60, 15),
    ('Brake fluid replacement', 'Periodic Service', 80, 150, 30),
    ('Coolant replacement', 'Periodic Service', 100, 200, 45),
    ('Spark plug replacement', 'Periodic Service', 60, 150, 30),
    ('Glow plug replacement', 'Periodic Service', 100, 300, 60),
    ('Brake pads replacement front', 'Brakes', 100, 200, 45),
    ('Brake pads replacement rear', 'Brakes', 80, 180, 45),
    ('Brake discs and pads replacement front', 'Brakes', 200, 400, 60),
    ('Brake discs and pads replacement rear', 'Brakes', 180, 350, 60),
    ('Brake shoes replacement', 'Brakes', 100, 200, 60),
    ('Tyre change (4 pcs)', 'Tyres', 80, 160, 45),
    ('Wheel balancing (4 pcs)', 'Tyres', 40, 80, 30),
    ('Tyre storage (season)', 'Tyres', 60, 120, 15),
    ('Tyre repair', 'Tyres', 20, 50, 20),
    ('Wheel alignment', 'Tyres', 100, 200, 45),
    ('Front shock absorber replacement', 'Suspension', 200, 500, 120),
    ('Rear shock absorber replacement', 'Suspension', 150, 400, 90),
    ('Control arm replacement', 'Suspension', 150, 350, 90),
    ('Stabiliser link replacement', 'Suspension', 50, 120, 30),
    ('Tie rod end replacement', 'Suspension', 80, 180, 45),
    ('Timing belt replacement', 'Timing', 400, 1200, 240),
    ('Timing kit with water pump replacement', 'Timing', 600, 1800, 300),
    ('Multi-V belt replacement', 'Timing', 80, 200, 45),
    ('Clutch replacement', 'Clutch', 500, 1500, 360),
    ('Dual mass flywheel replacement', 'Clutch', 800, 2500, 420),
    ('Starter motor replacement', 'Electrical System', 200, 500, 90),
    ('Alternator replacement', 'Electrical System', 250, 600, 90),
    ('Computer diagnostics', 'Diagnostics', 50, 150, 30),
    ('Error code clearing', 'Diagnostics', 30, 80, 15),
    ('Air conditioning check', 'Air Conditioning', 50, 100, 30),
    ('Air conditioning service', 'Air Conditioning', 150, 350, 60),
    ('Interior ozone treatment', 'Air Conditioning', 50, 100, 30),
    ('Muffler replacement', 'Exhaust System', 150, 400, 60),
    ('Catalytic converter replacement', 'Exhaust System', 500, 2000, 120),
    ('Exhaust welding', 'Exhaust System', 50, 150, 30),
    ('Battery replacement', 'Electrics', 30, 60, 15),
    ('Bulb replacement', 'Electrics', 20, 80, 15),
    ('Panel painting', 'Bodywork', 300, 1500, 480),
    ('Bodywork and paint repair', 'Bodywork', 500, 5000, 960),
    ('Paint polishing', 'Bodywork', 200, 600, 240),
    ('PDR dent removal', 'Bodywork', 100, 500, 120),
    ('Technical inspection', 'Inspection', 99, 99, 30),
    ('Technical inspection + emissions test', 'Inspection', 162, 162, 45),
    ('Engine cleaning', 'Other', 80, 200, 60),
    ('Chassis anti-corrosion treatment', 'Other', 200, 600, 120),
]

PAYMENT_METHODS = ['cash', 'card', 'bank_transfer', 'BLIK', 'leasing', 'instalments']
PAYMENT_WEIGHTS = [0.15, 0.40, 0.20, 0.15, 0.05, 0.05]

WORK_ORDER_STATUSES = ['new', 'in_progress', 'waiting_for_parts', 'completed', 'cancelled']
STATUS_WEIGHTS = [0.02, 0.03, 0.01, 0.92, 0.02]

LOCATION_TYPES = ['workshop', 'shop', 'workshop_and_shop']
LOCATION_TYPE_WEIGHTS = [0.30, 0.20, 0.50]

POSITIONS = {
    'workshop': ['mechanic', 'senior_mechanic', 'auto_electrician', 'panel_beater', 'painter', 'diagnostician'],
    'shop': ['sales_assistant', 'senior_sales_assistant', 'cashier', 'warehouse_operative'],
    'management': ['branch_manager', 'deputy_manager', 'accountant'],
}

print(f'Loaded reference data:')
print(f'  - {len(CITIES)} cities')
print(f'  - {len(CAR_MAKES)} car makes')
print(f'  - {sum(len(v) for v in PRODUCT_CATEGORIES.values())} products in {len(PRODUCT_CATEGORIES)} categories')
print(f'  - {len(SERVICE_CATALOGUE)} workshop services')

In [0]:
# ============================================================
# TABLE SCHEMAS  (column -> Spark SQL type)
# Use these to enforce correct types when reading parquet files:
#   schema = StructType([StructField(c, t) for c, t in build_schema(TABLE_SCHEMAS['dim_locations'])])
# or pass directly to spark.read.schema(ddl_string).parquet(path)
# ============================================================

from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType, DoubleType,
    BooleanType, DateType, TimestampType
)

_TYPE_MAP = {
    'STRING':    StringType(),
    'BIGINT':       IntegerType(),
    'BIGINT':    LongType(),
    'DOUBLE':    DoubleType(),
    'BOOLEAN':   BooleanType(),
    'DATE':      DateType(),
    'TIMESTAMP': TimestampType(),
}

def build_spark_schema(schema_dict):
    """Converts a schema dict {col: type_string} to a Spark StructType."""
    return StructType([
        StructField(col, _TYPE_MAP[typ], nullable=True)
        for col, typ in schema_dict.items()
    ])

def schema_to_ddl(schema_dict):
    """Converts a schema dict to a DDL string usable in spark.read.schema()."""
    return ', '.join(f'`{col}` {typ}' for col, typ in schema_dict.items())


# ------------------------------------------------------------------
# DIMENSION SCHEMAS
# ------------------------------------------------------------------

DIM_LOCATIONS_SCHEMA = {
    'location_id':     'BIGINT',
    'location_code':   'STRING',
    'name':            'STRING',
    'type':            'STRING',
    'street':          'STRING',
    'city':            'STRING',
    'region':          'STRING',
    'postal_code':     'STRING',
    'latitude':        'DOUBLE',
    'longitude':       'DOUBLE',
    'phone':           'STRING',
    'email':           'STRING',
    'manager_id':      'BIGINT',
    'number_of_bays':  'BIGINT',
    'area_m2':         'BIGINT',
    'opening_date':    'DATE',
    'is_active':       'BOOLEAN',
}

DIM_EMPLOYEES_SCHEMA = {
    'employee_id':        'BIGINT',
    'employee_code':      'STRING',
    'first_name':         'STRING',
    'last_name':          'STRING',
    'national_id':        'STRING',
    'position':           'STRING',
    'location_id':        'BIGINT',
    'hire_date':          'DATE',
    'termination_date':   'DATE',
    'hourly_rate':        'DOUBLE',
    'is_active':          'BOOLEAN',
}

DIM_CUSTOMERS_SCHEMA = {
    'customer_id':           'BIGINT',
    'customer_code':         'STRING',
    'customer_type':         'STRING',
    'first_name':            'STRING',
    'last_name':             'STRING',
    'company_name':          'STRING',
    'tax_id':                'STRING',
    'email':                 'STRING',
    'phone':                 'STRING',
    'city':                  'STRING',
    'postal_code':           'STRING',
    'registration_date':     'DATE',
    'preferred_location_id': 'BIGINT',
    'marketing_consent':     'BOOLEAN',
}

DIM_VEHICLES_SCHEMA = {
    'vehicle_id':              'BIGINT',
    'customer_id':             'BIGINT',
    'make':                    'STRING',
    'model':                   'STRING',
    'year':                    'BIGINT',
    'vin':                     'STRING',
    'registration_number':     'STRING',
    'fuel_type':               'STRING',
    'engine_displacement':     'DOUBLE',
    'horsepower':              'BIGINT',
    'color':                   'STRING',
    'mileage_km':              'BIGINT',
    'first_registration_date': 'DATE',
}

DIM_PRODUCTS_SCHEMA = {
    'product_id':          'BIGINT',
    'product_code':        'STRING',
    'name':                'STRING',
    'category':            'STRING',
    'manufacturer':        'STRING',
    'purchase_price_net':  'DOUBLE',
    'sale_price_net':      'DOUBLE',
    'vat_rate':            'BIGINT',
    'unit':                'STRING',
    'weight_kg':           'DOUBLE',
    'min_stock_level':     'BIGINT',
    'is_active':           'BOOLEAN',
}

DIM_SERVICES_SCHEMA = {
    'service_id':          'BIGINT',
    'service_code':        'STRING',
    'name':                'STRING',
    'category':            'STRING',
    'min_price_net':       'BIGINT',
    'max_price_net':       'BIGINT',
    'estimated_time_min':  'BIGINT',
    'is_active':           'BOOLEAN',
}

DIM_SUPPLIERS_SCHEMA = {
    'supplier_id':        'BIGINT',
    'supplier_code':      'STRING',
    'name':               'STRING',
    'tax_id':             'STRING',
    'city':               'STRING',
    'address':            'STRING',
    'postal_code':        'STRING',
    'phone':              'STRING',
    'email':              'STRING',
    'contact_person':     'STRING',
    'payment_terms_days': 'BIGINT',
    'min_order_value':    'DOUBLE',
    'is_active':          'BOOLEAN',
}

# ------------------------------------------------------------------
# FACT SCHEMAS
# ------------------------------------------------------------------

FACT_WORK_ORDERS_SCHEMA = {
    'work_order_id':       'BIGINT',
    'work_order_code':     'STRING',
    'location_id':         'BIGINT',
    'customer_id':         'BIGINT',
    'vehicle_id':          'BIGINT',
    'mechanic_id':         'BIGINT',
    'reception_date':      'DATE',
    'completion_date':     'DATE',
    'status':              'STRING',
    'mileage_at_reception':'BIGINT',
    'customer_notes':      'STRING',
    'year':                'BIGINT',
    'month':               'BIGINT',
}

FACT_WORK_ORDER_ITEMS_SCHEMA = {
    'wo_item_id':      'BIGINT',
    'work_order_id':   'BIGINT',
    'item_type':       'STRING',
    'service_id':      'BIGINT',
    'product_id':      'BIGINT',
    'quantity':        'BIGINT',
    'unit_price_net':  'DOUBLE',
    'value_net':       'DOUBLE',
    'vat_rate':        'BIGINT',
    'value_gross':     'DOUBLE',
    'discount_percent':'BIGINT',
}

FACT_SALES_TRANSACTIONS_SCHEMA = {
    'transaction_id':   'BIGINT',
    'transaction_code': 'STRING',
    'location_id':      'BIGINT',
    'customer_id':      'BIGINT',
    'employee_id':      'BIGINT',
    'transaction_date': 'TIMESTAMP',
    'payment_method':   'STRING',
    'receipt_number':   'STRING',
    'year':             'BIGINT',
    'month':            'BIGINT',
}

FACT_SALES_ITEMS_SCHEMA = {
    'sales_item_id':   'BIGINT',
    'transaction_id':  'BIGINT',
    'product_id':      'BIGINT',
    'quantity':        'BIGINT',
    'unit_price_net':  'DOUBLE',
    'discount_percent':'BIGINT',
    'value_net':       'DOUBLE',
    'vat_rate':        'BIGINT',
    'value_gross':     'DOUBLE',
}

FACT_INVOICES_SCHEMA = {
    'invoice_id':       'BIGINT',
    'invoice_code':     'STRING',
    'document_type':    'STRING',
    'source_type':      'STRING',
    'source_id':        'BIGINT',
    'customer_id':      'BIGINT',
    'location_id':      'BIGINT',
    'issue_date':       'DATE',
    'sale_date':        'DATE',
    'payment_due_date': 'DATE',
    'value_net':        'DOUBLE',
    'value_vat':        'DOUBLE',
    'value_gross':      'DOUBLE',
    'status':           'STRING',
    'year':             'BIGINT',
    'month':            'BIGINT',
}

FACT_PAYMENTS_SCHEMA = {
    'payment_id':         'BIGINT',
    'invoice_id':         'BIGINT',
    'payment_date':       'DATE',
    'amount':             'DOUBLE',
    'payment_method':     'STRING',
    'status':             'STRING',
    'transaction_number': 'STRING',
    'year':               'BIGINT',
    'month':              'BIGINT',
}

FACT_INVENTORY_MOVEMENTS_SCHEMA = {
    'movement_id':     'BIGINT',
    'product_id':      'BIGINT',
    'location_id':     'BIGINT',
    'movement_type':   'STRING',
    'quantity':        'BIGINT',
    'movement_date':   'DATE',
    'source_document': 'STRING',
    'document_number': 'STRING',
    'value_net':       'DOUBLE',
    'notes':           'STRING',
    'year':            'BIGINT',
    'month':           'BIGINT',
}

FACT_APPOINTMENTS_SCHEMA = {
    'appointment_id':   'BIGINT',
    'customer_id':      'BIGINT',
    'vehicle_id':       'BIGINT',
    'location_id':      'BIGINT',
    'service_id':       'BIGINT',
    'booking_date':     'DATE',
    'appointment_date': 'TIMESTAMP',
    'status':           'STRING',
    'booking_channel':  'STRING',
    'notes':            'STRING',
    'year':             'BIGINT',
    'month':            'BIGINT',
}

FACT_PURCHASE_ORDERS_SCHEMA = {
    'po_id':                  'BIGINT',
    'po_code':                'STRING',
    'supplier_id':            'BIGINT',
    'location_id':            'BIGINT',
    'order_date':             'DATE',
    'planned_delivery_date':  'DATE',
    'actual_delivery_date':   'DATE',
    'value_net':              'DOUBLE',
    'value_gross':            'DOUBLE',
    'status':                 'STRING',
    'year':                   'BIGINT',
}

FACT_PURCHASE_ORDER_ITEMS_SCHEMA = {
    'po_item_id':         'BIGINT',
    'po_id':              'BIGINT',
    'product_id':         'BIGINT',
    'quantity_ordered':   'BIGINT',
    'quantity_delivered': 'BIGINT',
    'unit_price_net':     'DOUBLE',
    'value_net':          'DOUBLE',
}

FACT_CUSTOMER_FEEDBACK_SCHEMA = {
    'feedback_id':   'BIGINT',
    'customer_id':   'BIGINT',
    'location_id':   'BIGINT',
    'work_order_id': 'BIGINT',
    'feedback_date': 'DATE',
    'rating':        'BIGINT',
    'comment':       'STRING',
    'category':      'STRING',
    'channel':       'STRING',
}

FACT_LOYALTY_PROGRAM_SCHEMA = {
    'loyalty_id':    'BIGINT',
    'customer_id':   'BIGINT',
    'event_date':    'DATE',
    'event_type':    'STRING',
    'points':        'BIGINT',
    'description':   'STRING',
    'balance_after': 'BIGINT',
    'tier':          'STRING',
}

FACT_EMPLOYEE_SCHEDULES_SCHEMA = {
    'schedule_id':    'BIGINT',
    'employee_id':    'BIGINT',
    'date':           'DATE',
    'start_hour':     'BIGINT',
    'end_hour':       'BIGINT',
    'shift_type':     'STRING',
    'overtime_hours': 'BIGINT',
    'attendance':     'STRING',
}

# ------------------------------------------------------------------
# Master registry  {table_name: schema_dict}
# ------------------------------------------------------------------
TABLE_SCHEMAS = {
    'dim_locations':             DIM_LOCATIONS_SCHEMA,
    'dim_employees':             DIM_EMPLOYEES_SCHEMA,
    'dim_customers':             DIM_CUSTOMERS_SCHEMA,
    'dim_vehicles':              DIM_VEHICLES_SCHEMA,
    'dim_products':              DIM_PRODUCTS_SCHEMA,
    'dim_services':              DIM_SERVICES_SCHEMA,
    'dim_suppliers':             DIM_SUPPLIERS_SCHEMA,
    'fact_work_orders':          FACT_WORK_ORDERS_SCHEMA,
    'fact_work_order_items':     FACT_WORK_ORDER_ITEMS_SCHEMA,
    'fact_sales_transactions':   FACT_SALES_TRANSACTIONS_SCHEMA,
    'fact_sales_items':          FACT_SALES_ITEMS_SCHEMA,
    'fact_invoices':             FACT_INVOICES_SCHEMA,
    'fact_payments':             FACT_PAYMENTS_SCHEMA,
    'fact_inventory_movements':  FACT_INVENTORY_MOVEMENTS_SCHEMA,
    'fact_appointments':         FACT_APPOINTMENTS_SCHEMA,
    'fact_purchase_orders':      FACT_PURCHASE_ORDERS_SCHEMA,
    'fact_purchase_order_items': FACT_PURCHASE_ORDER_ITEMS_SCHEMA,
    'fact_customer_feedback':    FACT_CUSTOMER_FEEDBACK_SCHEMA,
    'fact_loyalty_program':      FACT_LOYALTY_PROGRAM_SCHEMA,
    'fact_employee_schedules':   FACT_EMPLOYEE_SCHEDULES_SCHEMA,
}

print(f'Table schemas loaded: {len(TABLE_SCHEMAS)} tables')
print()
print('Usage examples:')
print('  spark.read.schema(schema_to_ddl(TABLE_SCHEMAS["dim_locations"])).parquet(path)')
print('  spark.read.schema(build_spark_schema(TABLE_SCHEMAS["fact_work_orders"])).parquet(path)')

## 2. Tabele wymiarowe (dimension tables)

In [0]:
# ============================================================
# dim_locations - 100 lokalizacji warsztatów/sklepów
# ============================================================
print('Generowanie dim_locations...')

locations = []
for i, (city, region, lat, lon) in enumerate(CITIES[:NUM_LOCATIONS]):
    loc_type = np.random.choice(LOCATION_TYPES, p=LOCATION_TYPE_WEIGHTS)
    opening = fake.date_between(start_date=date(2005, 1, 1), end_date=date(2020, 6, 30))
    locations.append({
        'location_id': i + 1,
        'location_code': f'LOC-{i+1:03d}',
        'name': f'AutoService {city}',
        'type': loc_type,
        'street': fake.street_address(),
        'city': city,
        'region': region,
        'postal_code': fake.postcode(),
        'latitude': lat + np.random.uniform(-0.02, 0.02),
        'longitude': lon + np.random.uniform(-0.02, 0.02),
        'phone': fake.phone_number(),
        'email': f'service.{city.lower().replace(" ", "").replace("-", "")}@autoservice.pl',
        'manager_id': None,  # to be filled after generating employees
        'number_of_bays': np.random.randint(4, 12) if loc_type != 'shop' else 0,
        'area_m2': np.random.randint(200, 800),
        'opening_date': opening,
        'is_active': True if i < 95 else False,  # 5 locations closed
    })

DIM_LOCATIONS_SCHEMA = {
    'location_id':     'BIGINT',
    'location_code':   'STRING',
    'name':            'STRING',
    'type':            'STRING',
    'street':          'STRING',
    'city':            'STRING',
    'region':          'STRING',
    'postal_code':     'STRING',
    'latitude':        'DOUBLE',
    'longitude':       'DOUBLE',
    'phone':           'STRING',
    'email':           'STRING',
    'manager_id':      'BIGINT',
    'number_of_bays':  'BIGINT',
    'area_m2':         'BIGINT',
    'opening_date':    'DATE',
    'is_active':       'BOOLEAN',
}

df_locations_pd = pd.DataFrame(locations)
df_locations = spark.createDataFrame(df_locations_pd, schema=schema_to_ddl(DIM_LOCATIONS_SCHEMA))
save_table_chunked(lambda n, offset: df_locations.toPandas().iloc[offset:offset+n],  'dim_locations', len(df_locations_pd), DIM_OUTPUT_DIR, )
df_locations.head()

In [0]:
# ============================================================
# dim_employees - pracownicy (~20 na lokalizację = ~2000)
# ============================================================
print('Generowanie dim_employees...')

employees = []
emp_id = 1

for _, loc in df_locations.iterrows():
    loc_id = loc['location_id']
    typ = loc['typ']
    
    # Kadra zarządzająca - zawsze
    for stanowisko in STANOWISKA['zarzadzanie']:
        employees.append({
            'employee_id': emp_id,
            'employee_code': f'EMP-{emp_id:05d}',
            'imie': fake.first_name(),
            'nazwisko': fake.last_name(),
            'pesel': fake.pesel(),
            'stanowisko': stanowisko,
            'location_id': loc_id,
            'data_zatrudnienia': fake.date_between(
                start_date=loc['data_otwarcia'],
                end_date=min(loc['data_otwarcia'] + timedelta(days=365), DATE_END)
            ),
            'data_zwolnienia': None,
            'stawka_godzinowa': round(np.random.uniform(45, 80), 2),
            'czy_aktywny': loc['czy_aktywna'],
        })
        emp_id += 1
    
    # Pracownicy warsztatu
    if typ in ('warsztat', 'warsztat_i_sklep'):
        n_mechanikow = np.random.randint(5, 10)
        for _ in range(n_mechanikow):
            stanowisko = random.choice(STANOWISKA['warsztat'])
            employees.append({
                'employee_id': emp_id,
                'employee_code': f'EMP-{emp_id:05d}',
                'imie': fake.first_name_male() if random.random() < 0.9 else fake.first_name_female(),
                'nazwisko': fake.last_name(),
                'pesel': fake.pesel(),
                'stanowisko': stanowisko,
                'location_id': loc_id,
                'data_zatrudnienia': fake.date_between(
                    start_date=loc['data_otwarcia'],
                    end_date=DATE_END
                ),
                'data_zwolnienia': fake.date_between(start_date=date(2000,1,1), end_date=DATE_END) if random.random() < 0.1 else None,
                'stawka_godzinowa': round(np.random.uniform(30, 65), 2),
                'czy_aktywny': random.random() > 0.1,
            })
            emp_id += 1
    
    # Pracownicy sklepu
    if typ in ('sklep', 'warsztat_i_sklep'):
        n_sprzedawcow = np.random.randint(3, 7)
        for _ in range(n_sprzedawcow):
            stanowisko = random.choice(STANOWISKA['sklep'])
            employees.append({
                'employee_id': emp_id,
                'employee_code': f'EMP-{emp_id:05d}',
                'imie': fake.first_name(),
                'nazwisko': fake.last_name(),
                'pesel': fake.pesel(),
                'stanowisko': stanowisko,
                'location_id': loc_id,
                'data_zatrudnienia': fake.date_between(
                    start_date=loc['data_otwarcia'],
                    end_date=DATE_END
                ),
                'data_zwolnienia': fake.date_between(start_date=date(2000,1,1), end_date=DATE_END) if random.random() < 0.15 else None,
                'stawka_godzinowa': round(np.random.uniform(25, 45), 2),
                'czy_aktywny': random.random() > 0.12,
            })
            emp_id += 1

DIM_EMPLOYEES_SCHEMA = {
    'employee_id':        'BIGINT',
    'employee_code':      'STRING',
    'first_name':         'STRING',
    'last_name':          'STRING',
    'national_id':        'STRING',
    'position':           'STRING',
    'location_id':        'BIGINT',
    'hire_date':          'DATE',
    'termination_date':   'DATE',
    'hourly_rate':        'DOUBLE',
    'is_active':          'BOOLEAN',
}


df_employees = pd.DataFrame(employees)
save_table_chunked(lambda n, offset: df_employees.iloc[offset:offset+n], 'dim_employees', len(df_employees), DIM_OUTPUT_DIR, )

# Lista ID mechaników i sprzedawców do użycia w tabelach faktowych
mechanic_ids = df_employees[df_employees['stanowisko'].isin(STANOWISKA['warsztat'])]['employee_id'].values
seller_ids = df_employees[df_employees['stanowisko'].isin(STANOWISKA['sklep'])]['employee_id'].values

# Mapowanie: location_id -> lista mechaników
loc_mechanics = df_employees[df_employees['stanowisko'].isin(STANOWISKA['warsztat'])].groupby('location_id')['employee_id'].apply(list).to_dict()
loc_sellers = df_employees[df_employees['stanowisko'].isin(STANOWISKA['sklep'])].groupby('location_id')['employee_id'].apply(list).to_dict()

print(f'  Mechanicy: {len(mechanic_ids)}, Sprzedawcy: {len(seller_ids)}')
df_employees.head()



In [0]:
# ============================================================
# dim_customers - klienci (4_500_000 * SCALE_FACTOR)
# ============================================================
from pyspark.sql.types import *

NUM_CUSTOMERS = int(5 * SCALE_FACTOR)
print(f'Generowanie dim_customers ({NUM_CUSTOMERS:,} klientów)...')

customer_types = np.random.choice(
    ['indywidualny', 'firma'], size=NUM_CUSTOMERS, p=[0.7, 0.3]
)

df_customers_pd = pd.DataFrame({
    'customer_id': np.arange(1, NUM_CUSTOMERS + 1),
    'customer_code': [f'CUS-{i:07d}' for i in range(1, NUM_CUSTOMERS + 1)],
    'typ_klienta': customer_types,
    'imie': [fake.first_name() if t == 'indywidualny' else '' for t in customer_types],
    'nazwisko': [fake.last_name() if t == 'indywidualny' else '' for t in customer_types],
    'nazwa_firmy': [fake.company() if t == 'firma' else '' for t in customer_types],
    'nip': [fake.company_vat() if t == 'firma' else '' for t in customer_types],
    'email': [fake.email() for _ in range(NUM_CUSTOMERS)],
    'telefon': [fake.phone_number() for _ in range(NUM_CUSTOMERS)],
    'miasto': np.random.choice([m[0] for m in MIASTA], size=NUM_CUSTOMERS),
    'kod_pocztowy': [fake.postcode() for _ in range(NUM_CUSTOMERS)],
    'data_rejestracji': random_dates(DATE_START, DATE_END, NUM_CUSTOMERS),
    'preferowana_lokalizacja_id': np.random.randint(1, NUM_LOCATIONS + 1, size=NUM_CUSTOMERS),
    'zgoda_marketing': np.random.choice([True, False], size=NUM_CUSTOMERS, p=[0.6, 0.4]),
})
 
df_customers = spark.createDataFrame(df_customers_pd, schema = StructType([
    StructField('customer_id', IntegerType()),
    StructField('customer_code', StringType()),
    StructField('typ_klienta', StringType()),
    StructField('imie', StringType()),
    StructField('nazwisko', StringType()),
    StructField('nazwa_firmy', StringType()),
    StructField('nip', StringType()),
    StructField('email', StringType()),
    StructField('telefon', StringType()),
    StructField('miasto', StringType()),
    StructField('kod_pocztowy', StringType()),
    StructField('data_rejestracji', DateType()),
    StructField('preferowana_lokalizacja_id', IntegerType()),
    StructField('zgoda_marketing', BooleanType())]))

# df_customers = customers
save_table_chunked(lambda n, offset: df_customers.toPandas().iloc[offset:offset+n], 'dim_customers', NUM_CUSTOMERS, DIM_OUTPUT_DIR, )
customer_ids = df_customers['customer_id'].values
df_customers.head()

In [0]:
# ============================================================
# dim_vehicles - pojazdy klientów (600K * SCALE_FACTOR)
# ============================================================
NUM_VEHICLES = int(1_600_000 * SCALE_FACTOR)
print(f'Generowanie dim_vehicles ({NUM_VEHICLES:,} pojazdów)...')

marki = list(MARKI_SAMOCHODOW.keys())
wagi = [MARKA_WAGI[m] for m in marki]
wagi_norm = np.array(wagi) / sum(wagi)

chosen_marki = np.random.choice(marki, size=NUM_VEHICLES, p=wagi_norm)
chosen_modele = [random.choice(MARKI_SAMOCHODOW[m]) for m in chosen_marki]

df_vehicles = pd.DataFrame({
    'vehicle_id': np.arange(1, NUM_VEHICLES + 1),
    'customer_id': np.random.choice(customer_ids, size=NUM_VEHICLES),
    'marka': chosen_marki,
    'model': chosen_modele,
    'rocznik': np.random.randint(2005, 2025, size=NUM_VEHICLES),
    'vin': [fake.bothify('???#########??????').upper() for _ in range(NUM_VEHICLES)],
    'nr_rejestracyjny': [fake.license_plate() for _ in range(NUM_VEHICLES)],
    'typ_paliwa': np.random.choice(TYPY_PALIWA, size=NUM_VEHICLES, p=PALIWO_WAGI),
    'pojemnosc_silnika': np.random.choice(
        [1.0, 1.2, 1.4, 1.5, 1.6, 1.8, 2.0, 2.2, 2.5, 3.0],
        size=NUM_VEHICLES,
        p=[0.05, 0.10, 0.15, 0.12, 0.18, 0.12, 0.12, 0.06, 0.05, 0.05]
    ),
    'moc_km': np.random.randint(60, 350, size=NUM_VEHICLES),
    'kolor': np.random.choice(KOLORY, size=NUM_VEHICLES),
    'przebieg_km': np.random.randint(5000, 350000, size=NUM_VEHICLES),
    'data_pierwszej_rejestracji': random_dates(date(2005,1,1), DATE_END, NUM_VEHICLES),
})

DIM_VEHICLES_SCHEMA = {
    'vehicle_id':              'BIGINT',
    'customer_id':             'BIGINT',
    'make':                    'STRING',
    'model':                   'STRING',
    'year':                    'BIGINT',
    'vin':                     'STRING',
    'registration_number':     'STRING',
    'fuel_type':               'STRING',
    'engine_displacement':     'DOUBLE',
    'horsepower':              'BIGINT',
    'color':                   'STRING',
    'mileage_km':              'BIGINT',
    'first_registration_date': 'DATE',
}
# df_vehicles = vehicles
save_table_chunked(lambda n, offset: df_vehicles.iloc[offset:offset+n], 'dim_vehicles', len(df_vehicles), DIM_OUTPUT_DIR, )
vehicle_ids = df_vehicles['vehicle_id'].values
print(f'  Średnio {NUM_VEHICLES / NUM_CUSTOMERS:.1f} pojazdów na klienta')
df_vehicles.head()

In [0]:
# ============================================================
# dim_products - produkty/części (~15K z wariantami)
# ============================================================
print('Generowanie dim_products...')

products = []
prod_id = 1

for kategoria, produkty_lista in KATEGORIE_PRODUKTOW.items():
    for nazwa_bazowa in produkty_lista:
        # Każdy produkt ma kilka wariantów (różne marki producenta)
        producenci = random.sample(
            ['Bosch', 'Continental', 'Valeo', 'Hella', 'Mann', 'Mahle', 'NGK',
             'Brembo', 'TRW', 'KYB', 'Monroe', 'Sachs', 'LuK', 'Gates',
             'SKF', 'Dayco', 'Castrol', 'Mobil', 'Shell', 'Total', 'Motul',
             'Liqui Moly', 'K2', 'Meguiars', 'Sonax', 'Goodyear', 'Michelin',
             'Continental', 'Bridgestone', 'Pirelli', 'Varta', 'Exide', 'Banner'],
            k=min(random.randint(2, 6), 32)
        )
        for producent in producenci:
            cena_bazowa = round(np.random.uniform(5, 800), 2)
            # Droższe produkty: opony, akumulatory, sprzęgło
            if 'Opona' in nazwa_bazowa:
                cena_bazowa = round(np.random.uniform(180, 600), 2)
            elif 'Akumulator' in nazwa_bazowa:
                cena_bazowa = round(np.random.uniform(250, 800), 2)
            elif 'Komplet sprzęgła' in nazwa_bazowa or 'Koło dwumasowe' in nazwa_bazowa:
                cena_bazowa = round(np.random.uniform(400, 2000), 2)
            elif 'Amortyzator' in nazwa_bazowa:
                cena_bazowa = round(np.random.uniform(100, 400), 2)
            elif 'Filtr' in nazwa_bazowa:
                cena_bazowa = round(np.random.uniform(15, 80), 2)
            elif 'Klocki' in nazwa_bazowa or 'Tarcze' in nazwa_bazowa:
                cena_bazowa = round(np.random.uniform(60, 300), 2)
            elif 'Olej' in nazwa_bazowa:
                cena_bazowa = round(np.random.uniform(30, 180), 2)
            elif 'Żarówka' in nazwa_bazowa:
                cena_bazowa = round(np.random.uniform(8, 120), 2)
            
            marza = round(np.random.uniform(1.15, 1.45), 2)
            
            products.append({
                'product_id': prod_id,
                'product_code': f'PRD-{prod_id:06d}',
                'nazwa': f'{nazwa_bazowa} {producent}',
                'kategoria': kategoria,
                'producent': producent,
                'cena_zakupu_netto': cena_bazowa,
                'cena_sprzedazy_netto': round(cena_bazowa * marza, 2),
                'vat_procent': 23,
                'jednostka': 'szt' if 'Olej' not in nazwa_bazowa and 'Płyn' not in nazwa_bazowa else 'L',
                'waga_kg': round(np.random.uniform(0.1, 15), 2),
                'min_stan_magazynowy': np.random.randint(2, 20),
                'czy_aktywny': random.random() > 0.05,
            })
            prod_id += 1
DIM_PRODUCTS_SCHEMA = {
    'product_id':          'BIGINT',
    'product_code':        'STRING',
    'name':                'STRING',
    'category':            'STRING',
    'manufacturer':        'STRING',
    'purchase_price_net':  'DOUBLE',
    'sale_price_net':      'DOUBLE',
    'vat_rate':            'BIGINT',
    'unit':                'STRING',
    'weight_kg':           'DOUBLE',
    'min_stock_level':     'BIGINT',
    'is_active':           'BOOLEAN',
}
df_products = pd.DataFrame(products)
save_table_chunked(lambda n, offset: df_products.iloc[offset:offset+n], 'dim_products', len(df_products), DIM_OUTPUT_DIR, )
product_ids = df_products['product_id'].values
print(f'  Produktów: {len(df_products):,} w {len(KATEGORIE_PRODUKTOW)} kategoriach')
df_products.head()

In [0]:
# ============================================================
# dim_services - katalog usług warsztatowych
# ============================================================
print('Generowanie dim_services...')

services = []
for i, (nazwa, kategoria, min_c, max_c, czas) in enumerate(KATALOG_USLUG):
    services.append({
        'service_id': i + 1,
        'service_code': f'SRV-{i+1:03d}',
        'nazwa': nazwa,
        'kategoria': kategoria,
        'cena_min_netto': min_c,
        'cena_max_netto': max_c,
        'szacowany_czas_min': czas,
        'czy_aktywna': True,
    })
DIM_SERVICES_SCHEMA = {
    'service_id':          'BIGINT',
    'service_code':        'STRING',
    'name':                'STRING',
    'category':            'STRING',
    'min_price_net':       'BIGINT',
    'max_price_net':       'BIGINT',
    'estimated_time_min':  'BIGINT',
    'is_active':           'BOOLEAN',
}

df_services = pd.DataFrame(services)
save_table(df_services, 'dim_services')
service_ids = df_services['service_id'].values

# ============================================================
# dim_suppliers - dostawcy części
# ============================================================
print('Generowanie dim_suppliers...')
DIM_SUPPLIERS_SCHEMA = {
    'supplier_id':        'BIGINT',
    'supplier_code':      'STRING',
    'name':               'STRING',
    'tax_id':             'STRING',
    'city':               'STRING',
    'address':            'STRING',
    'postal_code':        'STRING',
    'phone':              'STRING',
    'email':              'STRING',
    'contact_person':     'STRING',
    'payment_terms_days': 'BIGINT',
    'min_order_value':    'DOUBLE',
    'is_active':          'BOOLEAN',
}

NUM_SUPPLIERS = 3000
suppliers = []
for i in range(NUM_SUPPLIERS):
    suppliers.append({
        'supplier_id': i + 1,
        'supplier_code': f'SUP-{i+1:04d}',
        'nazwa': fake.company(),
        'nip': fake.company_vat(),
        'miasto': random.choice([m[0] for m in MIASTA]),
        'adres': fake.street_address(),
        'kod_pocztowy': fake.postcode(),
        'telefon': fake.phone_number(),
        'email': fake.company_email(),
        'osoba_kontaktowa': fake.name(),
        'warunki_platnosci_dni': random.choice([14, 21, 30, 45, 60]),
        'min_wartosc_zamowienia': round(np.random.uniform(200, 2000), 2),
        'czy_aktywny': random.random() > 0.08,
    })

df_suppliers = pd.DataFrame(suppliers)
save_table_chunked(lambda n, offset: df_suppliers.iloc[offset:offset+n], 'dim_suppliers', len(df_suppliers), DIM_OUTPUT_DIR, )
supplier_ids = df_suppliers['supplier_id'].values

print(f'\n=== PODSUMOWANIE TABEL WYMIAROWYCH ===')
for name, df in [('dim_locations', df_locations), ('dim_employees', df_employees),
                  ('dim_customers', df_customers), ('dim_vehicles', df_vehicles),
                  ('dim_products', df_products), ('dim_services', df_services),
                  ('dim_suppliers', df_suppliers)]:
    print(f'  {name}: {len(df):,} wierszy')

## 3. Tabele faktowe - zlecenia warsztatowe

In [0]:
# ============================================================
# fact_work_orders - zlecenia warsztatowe (5M * SCALE_FACTOR)
# ============================================================
NUM_WORK_ORDERS = int(5_000_000 * SCALE_FACTOR)
print(f'Generowanie fact_work_orders ({NUM_WORK_ORDERS:,} zleceń)...')

# Lokalizacje z warsztatem
workshop_locs = df_locations[df_locations['typ'].isin(['warsztat', 'warsztat_i_sklep'])]['location_id'].values

def generate_work_orders_chunk(chunk_size, offset):
    dates = seasonal_dates(DATE_START, DATE_END, chunk_size)
    loc_ids = np.random.choice(workshop_locs, size=chunk_size)
    
    # Przypisz mechanika z danej lokalizacji
    mech_ids = []
    for lid in loc_ids:
        mechs = loc_mechanics.get(lid, mechanic_ids[:5])
        mech_ids.append(random.choice(mechs))
    
    return pd.DataFrame({
        'work_order_id': np.arange(offset + 1, offset + chunk_size + 1),
        'work_order_code': [f'WO-{i:08d}' for i in range(offset + 1, offset + chunk_size + 1)],
        'location_id': loc_ids,
        'customer_id': np.random.choice(customer_ids, size=chunk_size),
        'vehicle_id': np.random.choice(vehicle_ids, size=chunk_size),
        'mechanic_id': mech_ids,
        'data_przyjecia': dates,
        'data_zakonczenia': dates + pd.to_timedelta(np.random.randint(0, 5, size=chunk_size), unit='D'),
        'status': np.random.choice(STATUSY_ZLECEN, size=chunk_size, p=STATUSY_WAGI),
        'przebieg_przy_przyjęciu': np.random.randint(10000, 350000, size=chunk_size),
        'uwagi_klienta': np.random.choice(
            ['', 'Stuk przy hamowaniu', 'Silnik traci moc', 'Wyciek oleju',
             'Wymiana opon sezonowa', 'Przegląd okresowy', 'Klima nie chłodzi',
             'Kontrolka silnika', 'Hałas z zawieszenia', 'Wymiana klocków',
             'Przygotowanie do przeglądu', 'Wymiana oleju', 'Problem z rozrusznikiem',
             'Drgania kierownicy', 'Wymiana świec', ''],
            size=chunk_size
        ),
        'rok': dates.year,
        'miesiac': dates.month,
    })

FACT_WORK_ORDERS_SCHEMA = {
    'work_order_id':       'BIGINT',
    'work_order_code':     'STRING',
    'location_id':         'BIGINT',
    'customer_id':         'BIGINT',
    'vehicle_id':          'BIGINT',
    'mechanic_id':         'BIGINT',
    'reception_date':      'DATE',
    'completion_date':     'DATE',
    'status':              'STRING',
    'mileage_at_reception':'BIGINT',
    'customer_notes':      'STRING',
    'year':                'BIGINT',
    'month':               'BIGINT',
}
save_table_chunked(generate_work_orders_chunk, 'fact_work_orders', NUM_WORK_ORDERS, FACT_OUTPUT_DIR,
                   partition_cols=['rok', 'miesiac'])

In [0]:
# ============================================================
# fact_work_order_items - pozycje zleceń (15M * SCALE_FACTOR)
# Każde zlecenie ma 1-6 pozycji (usługa + ewentualnie części)
# ============================================================
NUM_WO_ITEMS = int(15_000_000 * SCALE_FACTOR)
print(f'Generowanie fact_work_order_items ({NUM_WO_ITEMS:,} pozycji)...')

def generate_wo_items_chunk(chunk_size, offset):
    wo_ids = np.random.randint(1, NUM_WORK_ORDERS + 1, size=chunk_size)
    # Losowy typ pozycji: usługa lub część
    typ_pozycji = np.random.choice(['usluga', 'czesc'], size=chunk_size, p=[0.4, 0.6])
    
    srv_ids = np.where(
        typ_pozycji == 'usluga',
        np.random.choice(service_ids, size=chunk_size),
        0
    )
    prod_ids = np.where(
        typ_pozycji == 'czesc',
        np.random.choice(product_ids, size=chunk_size),
        0
    )
    
    ilosc = np.where(typ_pozycji == 'usluga', 1, np.random.randint(1, 5, size=chunk_size))
    cena_netto = np.where(
        typ_pozycji == 'usluga',
        np.random.uniform(30, 2000, size=chunk_size),
        np.random.uniform(5, 500, size=chunk_size)
    )
    cena_netto = np.round(cena_netto, 2)
    
    return pd.DataFrame({
        'wo_item_id': np.arange(offset + 1, offset + chunk_size + 1),
        'work_order_id': wo_ids,
        'typ_pozycji': typ_pozycji,
        'service_id': srv_ids.astype(int),
        'product_id': prod_ids.astype(int),
        'ilosc': ilosc,
        'cena_jednostkowa_netto': cena_netto,
        'wartosc_netto': np.round(cena_netto * ilosc, 2),
        'vat_procent': 23,
        'wartosc_brutto': np.round(cena_netto * ilosc * 1.23, 2),
        'rabat_procent': np.random.choice([0, 0, 0, 5, 10, 15], size=chunk_size),
    })
FACT_WORK_ORDER_ITEMS_SCHEMA = {
    'wo_item_id':      'BIGINT',
    'work_order_id':   'BIGINT',
    'item_type':       'STRING',
    'service_id':      'BIGINT',
    'product_id':      'BIGINT',
    'quantity':        'BIGINT',
    'unit_price_net':  'DOUBLE',
    'value_net':       'DOUBLE',
    'vat_rate':        'BIGINT',
    'value_gross':     'DOUBLE',
    'discount_percent':'BIGINT',
}

save_table_chunked(generate_wo_items_chunk, 'fact_work_order_items', NUM_WO_ITEMS, FACT_OUTPUT_DIR)

## 4. Tabele faktowe - sprzedaż sklepowa

In [0]:
# ============================================================
# fact_sales_transactions - transakcje sprzedaży sklepowej (30M * SCALE_FACTOR)
# ============================================================
NUM_SALES = int(30_000_000 * SCALE_FACTOR)
print(f'Generowanie fact_sales_transactions ({NUM_SALES:,} transakcji)...')

# Lokalizacje ze sklepem
shop_locs = df_locations[df_locations['typ'].isin(['sklep', 'warsztat_i_sklep'])]['location_id'].values

def generate_sales_chunk(chunk_size, offset):
    dates = seasonal_dates(DATE_START, DATE_END, chunk_size)
    hours = np.random.choice(range(7, 20), size=chunk_size, 
                              p=[0.03, 0.08, 0.10, 0.10, 0.09, 0.08, 0.08,
                                 0.08, 0.08, 0.08, 0.08, 0.07, 0.05])
    minutes = np.random.randint(0, 60, size=chunk_size)
    
    timestamps = dates + pd.to_timedelta(hours, unit='h') + pd.to_timedelta(minutes, unit='m')
    loc_ids = np.random.choice(shop_locs, size=chunk_size)
    
    seller_arr = []
    for lid in loc_ids:
        sellers = loc_sellers.get(lid, seller_ids[:3])
        seller_arr.append(random.choice(sellers))
    
    # ~70% transakcji ma klienta zarejestrowanego, ~30% to walk-in
    has_customer = np.random.random(size=chunk_size) < 0.7
    cust_ids = np.where(has_customer, np.random.choice(customer_ids, size=chunk_size), 0)
    
    return pd.DataFrame({
        'transaction_id': np.arange(offset + 1, offset + chunk_size + 1),
        'transaction_code': [f'TRX-{i:09d}' for i in range(offset + 1, offset + chunk_size + 1)],
        'location_id': loc_ids,
        'customer_id': cust_ids,
        'employee_id': seller_arr,
        'data_transakcji': timestamps,
        'metoda_platnosci': np.random.choice(METODY_PLATNOSCI, size=chunk_size, p=PLATNOSC_WAGI),
        'nr_paragonu': [f'PAR/{random.randint(1,999):03d}/{i+offset+1:08d}' for i in range(chunk_size)],
        'rok': dates.year,
        'miesiac': dates.month,
    })
FACT_SALES_TRANSACTIONS_SCHEMA = {
    'transaction_id':   'BIGINT',
    'transaction_code': 'STRING',
    'location_id':      'BIGINT',
    'customer_id':      'BIGINT',
    'employee_id':      'BIGINT',
    'transaction_date': 'TIMESTAMP',
    'payment_method':   'STRING',
    'receipt_number':   'STRING',
    'year':             'BIGINT',
    'month':            'BIGINT',
}

save_table_chunked(generate_sales_chunk, 'fact_sales_transactions', NUM_SALES,  FACT_OUTPUT_DIR,
                   partition_cols=['rok', 'miesiac'])

In [0]:
# ============================================================
# fact_sales_items - pozycje sprzedaży (90M * SCALE_FACTOR)
# Średnio 3 pozycje na transakcję
# ============================================================
NUM_SALES_ITEMS = int(90_000_000 * SCALE_FACTOR)
print(f'Generowanie fact_sales_items ({NUM_SALES_ITEMS:,} pozycji)...')

def generate_sales_items_chunk(chunk_size, offset):
    trx_ids = np.random.randint(1, NUM_SALES + 1, size=chunk_size)
    prod_ids_chunk = np.random.choice(product_ids, size=chunk_size)
    ilosc = np.random.choice([1, 1, 1, 2, 2, 3, 4], size=chunk_size)
    cena_netto = np.round(np.random.uniform(3, 600, size=chunk_size), 2)
    rabat = np.random.choice([0, 0, 0, 0, 5, 10, 15, 20], size=chunk_size)
    wartosc_po_rabacie = np.round(cena_netto * ilosc * (1 - rabat/100), 2)
    
    return pd.DataFrame({
        'sales_item_id': np.arange(offset + 1, offset + chunk_size + 1),
        'transaction_id': trx_ids,
        'product_id': prod_ids_chunk,
        'ilosc': ilosc,
        'cena_jednostkowa_netto': cena_netto,
        'rabat_procent': rabat,
        'wartosc_netto': wartosc_po_rabacie,
        'vat_procent': 23,
        'wartosc_brutto': np.round(wartosc_po_rabacie * 1.23, 2),
    })

FACT_SALES_ITEMS_SCHEMA = {
    'sales_item_id':   'BIGINT',
    'transaction_id':  'BIGINT',
    'product_id':      'BIGINT',
    'quantity':        'BIGINT',
    'unit_price_net':  'DOUBLE',
    'discount_percent':'BIGINT',
    'value_net':       'DOUBLE',
    'vat_rate':        'BIGINT',
    'value_gross':     'DOUBLE',
}
save_table_chunked(generate_sales_items_chunk, 'fact_sales_items', NUM_SALES_ITEMS, FACT_OUTPUT_DIR)

## 5. Tabele faktowe - faktury, płatności, magazyn

In [0]:
# ============================================================
# fact_invoices - faktury (35M * SCALE_FACTOR)
# Faktury powiązane z work_orders i sales_transactions
# ============================================================
NUM_INVOICES = int(35_000_000 * SCALE_FACTOR)
print(f'Generowanie fact_invoices ({NUM_INVOICES:,} faktur)...')

def generate_invoices_chunk(chunk_size, offset):
    dates = seasonal_dates(DATE_START, DATE_END, chunk_size)
    
    # ~15% faktur to faktury za zlecenia warsztatowe, ~85% za sprzedaż sklepową
    source_type = np.random.choice(
        ['work_order', 'sales'], size=chunk_size, p=[0.15, 0.85]
    )
    source_ids = np.where(
        source_type == 'work_order',
        np.random.randint(1, max(NUM_WORK_ORDERS, 1) + 1, size=chunk_size),
        np.random.randint(1, max(NUM_SALES, 1) + 1, size=chunk_size)
    )
    
    wartosc_netto = np.round(np.random.lognormal(mean=4.5, sigma=1.0, size=chunk_size), 2)
    wartosc_netto = np.clip(wartosc_netto, 10, 50000)
    wartosc_vat = np.round(wartosc_netto * 0.23, 2)
    
    # Typ: faktura VAT, paragon, faktura korygująca
    typ_dokumentu = np.random.choice(
        ['faktura_VAT', 'paragon', 'faktura_korygujaca'],
        size=chunk_size, p=[0.35, 0.60, 0.05]
    )
    
    return pd.DataFrame({
        'invoice_id': np.arange(offset + 1, offset + chunk_size + 1),
        'invoice_code': [f'FV/{dates[i].year}/{i+offset+1:08d}' for i in range(chunk_size)],
        'typ_dokumentu': typ_dokumentu,
        'source_type': source_type,
        'source_id': source_ids,
        'customer_id': np.random.choice(customer_ids, size=chunk_size),
        'location_id': np.random.randint(1, NUM_LOCATIONS + 1, size=chunk_size),
        'data_wystawienia': dates,
        'data_sprzedazy': dates - pd.to_timedelta(np.random.randint(0, 3, size=chunk_size), unit='D'),
        'termin_platnosci': dates + pd.to_timedelta(
            np.random.choice([0, 7, 14, 30], size=chunk_size, p=[0.5, 0.15, 0.2, 0.15]), unit='D'
        ),
        'wartosc_netto': wartosc_netto,
        'wartosc_vat': wartosc_vat,
        'wartosc_brutto': np.round(wartosc_netto + wartosc_vat, 2),
        'status': np.random.choice(
            ['oplacona', 'oczekuje', 'przeterminowana', 'anulowana'],
            size=chunk_size, p=[0.80, 0.10, 0.07, 0.03]
        ),
        'rok': dates.year,
        'miesiac': dates.month,
    })
FACT_INVOICES_SCHEMA = {
    'invoice_id':       'BIGINT',
    'invoice_code':     'STRING',
    'document_type':    'STRING',
    'source_type':      'STRING',
    'source_id':        'BIGINT',
    'customer_id':      'BIGINT',
    'location_id':      'BIGINT',
    'issue_date':       'DATE',
    'sale_date':        'DATE',
    'payment_due_date': 'DATE',
    'value_net':        'DOUBLE',
    'value_vat':        'DOUBLE',
    'value_gross':      'DOUBLE',
    'status':           'STRING',
    'year':             'BIGINT',
    'month':            'BIGINT',
}
save_table_chunked(generate_invoices_chunk, 'fact_invoices', NUM_INVOICES, FACT_OUTPUT_DIR,
                   partition_cols=['rok', 'miesiac'])

In [0]:
# ============================================================
# fact_payments - płatności (35M * SCALE_FACTOR)
# ============================================================
NUM_PAYMENTS = int(35_000_000 * SCALE_FACTOR)
print(f'Generowanie fact_payments ({NUM_PAYMENTS:,} płatności)...')

def generate_payments_chunk(chunk_size, offset):
    dates = seasonal_dates(DATE_START, DATE_END, chunk_size)
    kwota = np.round(np.random.lognormal(mean=4.5, sigma=1.0, size=chunk_size), 2)
    kwota = np.clip(kwota, 5, 60000)
    
    return pd.DataFrame({
        'payment_id': np.arange(offset + 1, offset + chunk_size + 1),
        'invoice_id': np.random.randint(1, max(NUM_INVOICES, 1) + 1, size=chunk_size),
        'data_platnosci': dates,
        'kwota': kwota,
        'metoda_platnosci': np.random.choice(METODY_PLATNOSCI, size=chunk_size, p=PLATNOSC_WAGI),
        'status': np.random.choice(
            ['zrealizowana', 'oczekuje', 'odrzucona', 'zwrot'],
            size=chunk_size, p=[0.90, 0.05, 0.03, 0.02]
        ),
        'numer_transakcji': [f'PAY-{uuid.uuid4().hex[:12].upper()}' for _ in range(chunk_size)],
        'rok': dates.year,
        'miesiac': dates.month,
    })
FACT_PAYMENTS_SCHEMA = {
    'payment_id':         'BIGINT',
    'invoice_id':         'BIGINT',
    'payment_date':       'DATE',
    'amount':             'DOUBLE',
    'payment_method':     'STRING',
    'status':             'STRING',
    'transaction_number': 'STRING',
    'year':               'BIGINT',
    'month':              'BIGINT',
}
save_table_chunked(generate_payments_chunk, 'fact_payments', NUM_PAYMENTS, FACT_OUTPUT_DIR,
                   partition_cols=['rok', 'miesiac'])

In [0]:
# ============================================================
# fact_inventory_movements - ruchy magazynowe (50M * SCALE_FACTOR)
# ============================================================
NUM_INVENTORY = int(50_000_000 * SCALE_FACTOR)
print(f'Generowanie fact_inventory_movements ({NUM_INVENTORY:,} ruchów)...')

def generate_inventory_chunk(chunk_size, offset):
    dates = random_dates(DATE_START, DATE_END, chunk_size)
    
    typ_ruchu = np.random.choice(
        ['przyjecie', 'wydanie_sprzedaz', 'wydanie_warsztat', 'zwrot', 'korekta', 'inwentaryzacja'],
        size=chunk_size, p=[0.25, 0.35, 0.25, 0.05, 0.05, 0.05]
    )
    
    ilosc = np.random.randint(1, 20, size=chunk_size)
    # Wydania mają ujemną ilość
    ilosc = np.where(
        np.isin(typ_ruchu, ['wydanie_sprzedaz', 'wydanie_warsztat']),
        -ilosc, ilosc
    )
    
    return pd.DataFrame({
        'movement_id': np.arange(offset + 1, offset + chunk_size + 1),
        'product_id': np.random.choice(product_ids, size=chunk_size),
        'location_id': np.random.randint(1, NUM_LOCATIONS + 1, size=chunk_size),
        'typ_ruchu': typ_ruchu,
        'ilosc': ilosc,
        'data_ruchu': dates,
        'dokument_zrodlowy': np.random.choice(
            ['PZ', 'WZ', 'RW', 'ZW', 'KOR', 'INW'],
            size=chunk_size
        ),
        'nr_dokumentu': [f'DOC-{i+offset+1:09d}' for i in range(chunk_size)],
        'wartosc_netto': np.round(np.abs(ilosc) * np.random.uniform(5, 500, size=chunk_size), 2),
        'uwagi': np.random.choice(['', '', '', 'Dostawa regularna', 'Zamówienie specjalne',
                                     'Zwrot od klienta', 'Korekta stanów', ''], size=chunk_size),
        'rok': dates.year,
        'miesiac': dates.month,
    })

FACT_INVENTORY_MOVEMENTS_SCHEMA = {
    'movement_id':     'BIGINT',
    'product_id':      'BIGINT',
    'location_id':     'BIGINT',
    'movement_type':   'STRING',
    'quantity':        'BIGINT',
    'movement_date':   'DATE',
    'source_document': 'STRING',
    'document_number': 'STRING',
    'value_net':       'DOUBLE',
    'notes':           'STRING',
    'year':            'BIGINT',
    'month':           'BIGINT',
}
save_table_chunked(generate_inventory_chunk, 'fact_inventory_movements', NUM_INVENTORY, FACT_OUTPUT_DIR,
                   partition_cols=['rok', 'miesiac'])

## 6. Tabele wspierające

In [0]:
# ============================================================
# fact_appointments - rezerwacje wizyt (5M * SCALE_FACTOR)
# ============================================================
NUM_APPOINTMENTS = int(5_000_000 * SCALE_FACTOR)
print(f'Generowanie fact_appointments ({NUM_APPOINTMENTS:,} rezerwacji)...')

def generate_appointments_chunk(chunk_size, offset):
    dates = seasonal_dates(DATE_START, DATE_END, chunk_size)
    hours = np.random.choice(range(7, 17), size=chunk_size)
    timestamps = dates + pd.to_timedelta(hours, unit='h')
    
    return pd.DataFrame({
        'appointment_id': np.arange(offset + 1, offset + chunk_size + 1),
        'customer_id': np.random.choice(customer_ids, size=chunk_size),
        'vehicle_id': np.random.choice(vehicle_ids, size=chunk_size),
        'location_id': np.random.choice(workshop_locs, size=chunk_size),
        'service_id': np.random.choice(service_ids, size=chunk_size),
        'data_rezerwacji': dates - pd.to_timedelta(np.random.randint(1, 14, size=chunk_size), unit='D'),
        'data_wizyty': timestamps,
        'status': np.random.choice(
            ['potwierdzona', 'zrealizowana', 'anulowana', 'niestawienie_sie'],
            size=chunk_size, p=[0.10, 0.75, 0.10, 0.05]
        ),
        'kanal_rezerwacji': np.random.choice(
            ['telefon', 'online', 'osobiscie', 'email'],
            size=chunk_size, p=[0.35, 0.40, 0.15, 0.10]
        ),
        'uwagi': np.random.choice(
            ['', '', '', 'Proszę o kontakt telefoniczny', 'Samochód zastępczy',
             'Preferuję rano', 'Pilne', 'Umówiony wcześniej', ''],
            size=chunk_size
        ),
        'rok': dates.year,
        'miesiac': dates.month,
    })
FACT_APPOINTMENTS_SCHEMA = {
    'appointment_id':   'BIGINT',
    'customer_id':      'BIGINT',
    'vehicle_id':       'BIGINT',
    'location_id':      'BIGINT',
    'service_id':       'BIGINT',
    'booking_date':     'DATE',
    'appointment_date': 'TIMESTAMP',
    'status':           'STRING',
    'booking_channel':  'STRING',
    'notes':            'STRING',
    'year':             'BIGINT',
    'month':            'BIGINT',
}
save_table_chunked(generate_appointments_chunk, 'fact_appointments', NUM_APPOINTMENTS,  FACT_OUTPUT_DIR,
                   partition_cols=['rok', 'miesiac'])

In [0]:
# ============================================================
# fact_purchase_orders - zamówienia do dostawców (500K * SCALE_FACTOR)
# ============================================================
NUM_PO = int(500_000 * SCALE_FACTOR)
print(f'Generowanie fact_purchase_orders ({NUM_PO:,} zamówień)...')

def generate_po_chunk(chunk_size, offset):
    dates = random_dates(DATE_START, DATE_END, chunk_size)
    wartosc = np.round(np.random.lognormal(mean=7, sigma=0.8, size=chunk_size), 2)
    wartosc = np.clip(wartosc, 200, 100000)
    
    return pd.DataFrame({
        'po_id': np.arange(offset + 1, offset + chunk_size + 1),
        'po_code': [f'PO-{i+offset+1:07d}' for i in range(chunk_size)],
        'supplier_id': np.random.choice(supplier_ids, size=chunk_size),
        'location_id': np.random.randint(1, NUM_LOCATIONS + 1, size=chunk_size),
        'data_zamowienia': dates,
        'data_dostawy_planowana': dates + pd.to_timedelta(np.random.randint(3, 21, size=chunk_size), unit='D'),
        'data_dostawy_rzeczywista': dates + pd.to_timedelta(np.random.randint(3, 25, size=chunk_size), unit='D'),
        'wartosc_netto': wartosc,
        'wartosc_brutto': np.round(wartosc * 1.23, 2),
        'status': np.random.choice(
            ['zlozono', 'w_realizacji', 'dostarczone', 'czesciowo_dostarczone', 'anulowane'],
            size=chunk_size, p=[0.03, 0.05, 0.85, 0.05, 0.02]
        ),
        'rok': dates.year,
    })
FACT_PURCHASE_ORDERS_SCHEMA = {
    'po_id':                  'BIGINT',
    'po_code':                'STRING',
    'supplier_id':            'BIGINT',
    'location_id':            'BIGINT',
    'order_date':             'DATE',
    'planned_delivery_date':  'DATE',
    'actual_delivery_date':   'DATE',
    'value_net':              'DOUBLE',
    'value_gross':            'DOUBLE',
    'status':                 'STRING',
    'year':                   'BIGINT',
}

save_table_chunked(generate_po_chunk, 'fact_purchase_orders', NUM_PO, FACT_OUTPUT_DIR)

# ============================================================
# fact_purchase_order_items - pozycje zamówień (2M * SCALE_FACTOR)
# ============================================================
NUM_PO_ITEMS = int(2_000_000 * SCALE_FACTOR)
print(f'Generowanie fact_purchase_order_items ({NUM_PO_ITEMS:,} pozycji)...')

def generate_po_items_chunk(chunk_size, offset):
    ilosc = np.random.randint(1, 50, size=chunk_size)
    cena = np.round(np.random.uniform(5, 500, size=chunk_size), 2)
    
    return pd.DataFrame({
        'po_item_id': np.arange(offset + 1, offset + chunk_size + 1),
        'po_id': np.random.randint(1, max(NUM_PO, 1) + 1, size=chunk_size),
        'product_id': np.random.choice(product_ids, size=chunk_size),
        'ilosc_zamowiona': ilosc,
        'ilosc_dostarczona': np.clip(ilosc + np.random.randint(-2, 1, size=chunk_size), 0, 100),
        'cena_jednostkowa_netto': cena,
        'wartosc_netto': np.round(cena * ilosc, 2),
    })
FACT_PURCHASE_ORDER_ITEMS_SCHEMA = {
    'po_item_id':         'BIGINT',
    'po_id':              'BIGINT',
    'product_id':         'BIGINT',
    'quantity_ordered':   'BIGINT',
    'quantity_delivered': 'BIGINT',
    'unit_price_net':     'DOUBLE',
    'value_net':          'DOUBLE',
}

save_table_chunked(generate_po_items_chunk, 'fact_purchase_order_items', NUM_PO_ITEMS, FACT_OUTPUT_DIR)

In [0]:
# ============================================================
# fact_customer_feedback - opinie klientów (2M * SCALE_FACTOR)
# ============================================================
NUM_FEEDBACK = int(2_000_000 * SCALE_FACTOR)
print(f'Generowanie fact_customer_feedback ({NUM_FEEDBACK:,} opinii)...')

KOMENTARZE = [
    'Bardzo profesjonalna obsługa', 'Szybka realizacja', 'Polecam!',
    'Trochę za drogo', 'Długi czas oczekiwania', 'Świetna komunikacja',
    'Fachowa naprawa', 'Samochód gotowy przed terminem', 'Miła obsługa',
    'Mogłoby być taniej', 'Wrócę na pewno', 'Solidna robota',
    'Uczciwe ceny', 'Problem wrócił po miesiącu', 'Brak uwag',
    'Rewelacja!', 'Przeciętnie', 'Do poprawy', 'OK', '',
]

def generate_feedback_chunk(chunk_size, offset):
    dates = random_dates(DATE_START, DATE_END, chunk_size)
    
    # Oceny z rozkładem: więcej pozytywnych
    oceny = np.random.choice(
        [1, 2, 3, 4, 5], size=chunk_size,
        p=[0.03, 0.05, 0.12, 0.30, 0.50]
    )
    
    return pd.DataFrame({
        'feedback_id': np.arange(offset + 1, offset + chunk_size + 1),
        'customer_id': np.random.choice(customer_ids, size=chunk_size),
        'location_id': np.random.randint(1, NUM_LOCATIONS + 1, size=chunk_size),
        'work_order_id': np.random.randint(1, max(NUM_WORK_ORDERS, 1) + 1, size=chunk_size),
        'data_opinii': dates,
        'ocena': oceny,
        'komentarz': np.random.choice(KOMENTARZE, size=chunk_size),
        'kategoria': np.random.choice(
            ['obsluga', 'jakosc_naprawy', 'czas_realizacji', 'cena', 'czystosc', 'ogolna'],
            size=chunk_size, p=[0.20, 0.25, 0.15, 0.15, 0.10, 0.15]
        ),
        'kanal': np.random.choice(
            ['google', 'formularz_online', 'email', 'telefon'],
            size=chunk_size, p=[0.40, 0.30, 0.20, 0.10]
        ),
    })
FACT_CUSTOMER_FEEDBACK_SCHEMA = {
    'feedback_id':   'BIGINT',
    'customer_id':   'BIGINT',
    'location_id':   'BIGINT',
    'work_order_id': 'BIGINT',
    'feedback_date': 'DATE',
    'rating':        'BIGINT',
    'comment':       'STRING',
    'category':      'STRING',
    'channel':       'STRING',
}
save_table_chunked(generate_feedback_chunk, 'fact_customer_feedback', NUM_FEEDBACK, FACT_OUTPUT_DIR)

In [0]:
# ============================================================
# fact_loyalty_program - program lojalnościowy (500K * SCALE_FACTOR)
# ============================================================
NUM_LOYALTY = int(500_000 * SCALE_FACTOR)
print(f'Generowanie fact_loyalty_program ({NUM_LOYALTY:,} wpisów)...')

def generate_loyalty_chunk(chunk_size, offset):
    dates = random_dates(date(2021, 1, 1), DATE_END, chunk_size)  # Program od 2021
    
    return pd.DataFrame({
        'loyalty_id': np.arange(offset + 1, offset + chunk_size + 1),
        'customer_id': np.random.choice(customer_ids, size=chunk_size),
        'data_zdarzenia': dates,
        'typ_zdarzenia': np.random.choice(
            ['naliczenie_punktow', 'wymiana_punktow', 'bonus', 'wygasniecie'],
            size=chunk_size, p=[0.60, 0.20, 0.10, 0.10]
        ),
        'punkty': np.random.choice(
            [-500, -200, -100, 10, 20, 50, 100, 200, 500],
            size=chunk_size, p=[0.05, 0.07, 0.08, 0.20, 0.25, 0.15, 0.10, 0.05, 0.05]
        ),
        'opis': np.random.choice(
            ['Zakup w sklepie', 'Usługa warsztatowa', 'Bonus powitalny',
             'Bonus urodzinowy', 'Wymiana na rabat 10%', 'Wymiana na rabat 20%',
             'Wymiana na darmowy przegląd', 'Punkty wygasłe', 'Polecenie znajomego'],
            size=chunk_size
        ),
        'saldo_po': np.random.randint(0, 5000, size=chunk_size),
        'poziom': np.random.choice(
            ['standard', 'silver', 'gold', 'platinum'],
            size=chunk_size, p=[0.50, 0.30, 0.15, 0.05]
        ),
    })
FACT_LOYALTY_PROGRAM_SCHEMA = {
    'loyalty_id':    'BIGINT',
    'customer_id':   'BIGINT',
    'event_date':    'DATE',
    'event_type':    'STRING',
    'points':        'BIGINT',
    'description':   'STRING',
    'balance_after': 'BIGINT',
    'tier':          'STRING',
}

save_table_chunked(generate_loyalty_chunk, 'fact_loyalty_program', NUM_LOYALTY, FACT_OUTPUT_DIR)

In [0]:
# ============================================================
# fact_employee_schedules - grafiki pracy (3M * SCALE_FACTOR)
# ============================================================
NUM_SCHEDULES = int(3_000_000 * SCALE_FACTOR)
print(f'Generowanie fact_employee_schedules ({NUM_SCHEDULES:,} wpisów)...')

all_employee_ids = df_employees['employee_id'].values

def generate_schedules_chunk(chunk_size, offset):
    dates = random_dates(DATE_START, DATE_END, chunk_size)
    
    godzina_start = np.random.choice([6, 7, 8, 9, 10, 12, 14], size=chunk_size,
                                       p=[0.05, 0.25, 0.30, 0.15, 0.05, 0.10, 0.10])
    czas_pracy = np.random.choice([4, 6, 8, 10, 12], size=chunk_size,
                                    p=[0.10, 0.10, 0.60, 0.15, 0.05])
    
    return pd.DataFrame({
        'schedule_id': np.arange(offset + 1, offset + chunk_size + 1),
        'employee_id': np.random.choice(all_employee_ids, size=chunk_size),
        'data': dates,
        'godzina_start': godzina_start,
        'godzina_koniec': godzina_start + czas_pracy,
        'typ_zmiany': np.random.choice(
            ['dzienna', 'poranna', 'popoludniowa', 'nocna', 'wolne', 'urlop', 'chorobowe'],
            size=chunk_size, p=[0.40, 0.15, 0.15, 0.02, 0.15, 0.08, 0.05]
        ),
        'nadgodziny_h': np.random.choice(
            [0, 0, 0, 0, 0, 1, 2, 3, 4],
            size=chunk_size
        ),
        'obecnosc': np.random.choice(
            ['obecny', 'nieobecny_usprawiedliwiony', 'nieobecny_nieusprawiedliwiony', 'spozniony'],
            size=chunk_size, p=[0.88, 0.08, 0.02, 0.02]
        ),
    })

FACT_EMPLOYEE_SCHEDULES_SCHEMA = {
    'schedule_id':    'BIGINT',
    'employee_id':    'BIGINT',
    'date':           'DATE',
    'start_hour':     'BIGINT',
    'end_hour':       'BIGINT',
    'shift_type':     'STRING',
    'overtime_hours': 'BIGINT',
    'attendance':     'STRING',
}
save_table_chunked(generate_schedules_chunk, 'fact_employee_schedules', NUM_SCHEDULES, FACT_OUTPUT_DIR,)

## 7. Walidacja i statystyki

In [0]:
# ============================================================
# WALIDACJA I PODSUMOWANIE
# ============================================================
import glob as glob_module

print('=' * 60)
print('PODSUMOWANIE GENERACJI DANYCH')
print('=' * 60)
print(f'SCALE_FACTOR: {SCALE_FACTOR}')
print(f'Katalog DIM: {os.path.abspath(DIM_OUTPUT_DIR)}')
print(f'Katalog FACT: {os.path.abspath(FACT_OUTPUT_DIR)}')
print()

total_size = 0
table_stats = []

for output_dir in [DIM_OUTPUT_DIR, FACT_OUTPUT_DIR]:
    for table_dir in sorted(os.listdir(output_dir)):
        table_path = os.path.join(output_dir, table_dir)
        if os.path.isdir(table_path):
            # Zlicz rozmiar plików
            size = 0
            file_count = 0
            for root, dirs, files in os.walk(table_path):
                for f in files:
                    fp = os.path.join(root, f)
                    size += os.path.getsize(fp)
                    file_count += 1

            size_mb = size / (1024 * 1024)
            total_size += size
            table_stats.append({
                'tabela': table_dir,
                'pliki': file_count,
                'rozmiar_MB': round(size_mb, 1),
            })

df_stats = pd.DataFrame(table_stats)
print(df_stats.to_string(index=False))
print()
print(f'ŁĄCZNY ROZMIAR: {total_size / (1024**3):.2f} GB')
print(f'Szacowany rozmiar przy SCALE_FACTOR=1.0: ~{total_size / (1024**3) / SCALE_FACTOR:.1f} GB')
print()
print('Gotowe! Dane znajdują się w katalogach:')
print(' ', os.path.abspath(DIM_OUTPUT_DIR))
print(' ', os.path.abspath(FACT_OUTPUT_DIR))
print()
print('Aby załadować dane do Databricks:')
print('  1. Wgraj katalogi dim/ oraz fact/ do DBFS lub Unity Catalog Volume')
print('  2. Użyj spark.read.parquet("dbfs:/path/to/tabela/") ')
print('  3. Lub CREATE TABLE ... USING PARQUET LOCATION ...')